# **Machine Learning for Vision and Multimedia project**

This notebook implements an image colorization pipeline using a Pix2Pix architecture.

## **Imports and device setup**
Import all necessary libraries and set up the device for computation.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchsummary import summary

from PIL import Image
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from skimage.color import rgb2lab, lab2rgb, deltaE_cie76
import numpy as np

from tqdm import tqdm
import glob
import re

from torchvision import utils as vutils
from torchvision.utils import save_image
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

import pandas as pd

import warnings
warnings.filterwarnings("ignore")

ngpu = 1
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

## **Custom dataset class definition**
Defines a custom PyTorch dataset class for loading and preprocessing images from the CelebA dataset. The images are resized, converted to the LAB color space, and split into lightness (L) and color (AB) channels, which are returned as tensors for training.

In [ ]:
class CelebADataset(Dataset):
    def __init__(self, root_dir, image_size=256, crop_type='center'):
        self.root_dir = root_dir
        self.image_size = image_size
        self.crop_type = crop_type.lower()
        self.image_paths = [os.path.join(root_dir, img_name) for img_name in os.listdir(root_dir)]
        self.pre_transform = transforms.Compose([
            transforms.Resize(self.image_size),
            transforms.CenterCrop(self.image_size)
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert('RGB')        
        img = self.pre_transform(img)

        # Convert from RGB to LAB
        img_lab = rgb2lab(np.array(img)).astype("float32")
        img_l = img_lab[:, :, 0:1] / 100.0             # [0, 1]
        img_ab = img_lab[:, :, 1:3] / 128.0            # [-1, 1]

        assert img_ab.shape[2] == 2, f"img_ab has shape {img_ab.shape}, expected 2 channels"

        # Convert in tensors
        img_l = torch.from_numpy(img_l).permute(2, 0, 1)     # (1, H, W)
        img_ab = torch.from_numpy(img_ab).permute(2, 0, 1)   # (2, H, W)

        return img_l, img_ab

## **Dataset path initialization**
Set the root directory for the dataset.

In [ ]:
dataroot= "/kaggle/input/celeba-short-dataset-training-validation-test/celebA_dataset"
print('Completed')

## **Dataset loader function**
Defines a helper function get_dataset() to load training, validation, or test splits of the CelebA dataset using the CelebADataset class and return corresponding PyTorch dataloaders.



In [ ]:
def get_dataset(i, dataroot, image_size=256, batch_size=16, shuffle=True):
    if i == 1:  # Training
        split_dir = os.path.join(dataroot, "train")   
    elif i == 2:  # Validation
        split_dir = os.path.join(dataroot, "validation")
    elif i == 3:  # Test
        split_dir = os.path.join(dataroot, "test")
    else:
        raise ValueError("Parameter 'i' not valid. Use 1 (training), 2 (validation) o 3 (test).")

    dataset = CelebADataset(root_dir=split_dir, image_size=256)

    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=4, pin_memory=True)

    return dataset, dataloader

train_dataset, train_loader = get_dataset(1, dataroot)
val_dataset, val_loader = get_dataset(2, dataroot, shuffle=False)
test_dataset, test_loader = get_dataset(3, dataroot, shuffle=False)

## **Dataset size display**
Prints the number of images loaded for training, validation, and test datasets to confirm correct dataset loading.

In [ ]:
print(f"Training: {len(train_dataset)} images")
print(f"Validation: {len(val_dataset)} images")
print(f"Test: {len(test_dataset)} images")

## **Visualizing grayscale and color image pairs**
Displays a set of sample image pairs from the training set, showing the grayscale input alongside the target color image reconstructed from LAB channels.

In [ ]:
def show_image_pairs(train_dataset, num_images=5):
    fig, axes = plt.subplots(num_images, 2, figsize=(8, num_images * 3))

    for i in range(num_images):
        img_l, img_ab = train_dataset[i]

        l = img_l.squeeze().numpy() * 100  
        ab = img_ab.numpy() * 128         

        lab = np.concatenate((l[np.newaxis, :, :], ab), axis=0)
        lab = lab.transpose(1, 2, 0)  
        img_rgb = lab2rgb(lab)

        axes[i, 0].imshow(l, cmap='gray')
        axes[i, 0].set_title('Input')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(img_rgb)
        axes[i, 1].set_title('Target')
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()
    plt.close()

show_image_pairs(train_dataset, num_images=5)

## **U-Net generator architecture**
efines a U-Net style generator network with downsampling and upsampling blocks. The architecture uses skip connections to combine encoder and decoder features, commonly used in image-to-image tasks like colorization.

In [ ]:
class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, down=True, use_batchnorm=True, dropout=False):
        super(UNetBlock, self).__init__()
        layers = []
        if down:
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False))
        else:
            layers.append(nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False))

        if use_batchnorm:
            layers.append(nn.BatchNorm2d(out_channels))

        if down:
            layers.append(nn.LeakyReLU(0.2))
        else:
            layers.append(nn.ReLU())

        if dropout:
            layers.append(nn.Dropout(0.5))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class UNetGenerator(nn.Module):
    def __init__(self, input_channels=1, output_channels=2, features=64):
        super(UNetGenerator, self).__init__()

        # Encoder (downsampling)
        self.down1 = UNetBlock(input_channels, features, down=True, use_batchnorm=False)  # 1 -> 64
        self.down2 = UNetBlock(features, features * 2)  # 64 -> 128
        self.down3 = UNetBlock(features * 2, features * 4)  # 128 -> 256
        self.down4 = UNetBlock(features * 4, features * 8)  # 256 -> 512
        self.down5 = UNetBlock(features * 8, features * 8)  # 512 -> 512
        self.down6 = UNetBlock(features * 8, features * 8)  # 512 -> 512
        self.down7 = UNetBlock(features * 8, features * 8)  # 512 -> 512
        self.bottleneck = UNetBlock(features * 8, features * 8, use_batchnorm=False)  # bottleneck

        # Decoder (upsampling)
        self.up1 = UNetBlock(features * 8, features * 8, down=False, dropout=True)  # skip with down7
        self.up2 = UNetBlock(features * 8 * 2, features * 8, down=False, dropout=True)  # skip with down6
        self.up3 = UNetBlock(features * 8 * 2, features * 8, down=False, dropout=True)  # skip with down5
        self.up4 = UNetBlock(features * 8 * 2, features * 8, down=False)  # skip with down4
        self.up5 = UNetBlock(features * 8 * 2, features * 4, down=False)  # skip with down3
        self.up6 = UNetBlock(features * 4 * 2, features * 2, down=False)  # skip with down2
        self.up7 = UNetBlock(features * 2 * 2, features, down=False)      # skip with down1

        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(features * 2, output_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        d5 = self.down5(d4)
        d6 = self.down6(d5)
        d7 = self.down7(d6)
        bottleneck = self.bottleneck(d7)

        up1 = self.up1(bottleneck)
        up2 = self.up2(torch.cat([up1, d7], dim=1))
        up3 = self.up3(torch.cat([up2, d6], dim=1))
        up4 = self.up4(torch.cat([up3, d5], dim=1))
        up5 = self.up5(torch.cat([up4, d4], dim=1))
        up6 = self.up6(torch.cat([up5, d3], dim=1))
        up7 = self.up7(torch.cat([up6, d2], dim=1))
        final = self.final_up(torch.cat([up7, d1], dim=1))
        return final

## **PatchGAN Discriminator**
Implements a PatchGAN-based discriminator network, which evaluates image realism at the patch level by classifying each patch in the concatenated L+AB image as real or fake.

In [ ]:
class PatchGANDiscriminator(nn.Module):
    def __init__(self, input_channels=3, features=[64, 128, 256, 512]):
        super(PatchGANDiscriminator, self).__init__()

        self.input_channels = input_channels
        layers = []
        in_channels = input_channels  

        layers.append(nn.Sequential(
            nn.Conv2d(in_channels, features[0], kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2)
        ))

        in_channels = features[0]
        for feature in features[1:]:
            layers.append(nn.Sequential(
                nn.Conv2d(in_channels, feature, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(feature),
                nn.LeakyReLU(0.2)
            ))
            in_channels = feature

        layers.append(nn.Conv2d(in_channels, 1, kernel_size=4, stride=1, padding=1))  # Output: patch

        self.model = nn.Sequential(*layers)

    def forward(self, input_l, ab):
      input_concat = torch.cat([input_l, ab], dim=1)
      return self.model(input_concat)

## **Weight initialization function**
Defines a utility function to initialize the weights of convolutional and batch normalization layers using a normal distribution, which helps in stabilizing GAN training.

In [ ]:
def initialize_weights(module):

    if isinstance(module, nn.Conv2d):
        nn.init.normal_(module.weight.data, mean=0.0, std=0.02) 

        if module.bias is not None:
            nn.init.constant_(module.bias.data, 0)

    elif isinstance(module, nn.BatchNorm2d):
        nn.init.normal_(module.weight.data, mean=1.0, std=0.02)
        nn.init.constant_(module.bias.data, 0) 

## **Model initialization and summary**
Creates instances of the generator and discriminator networks, applies weight initialization, moves the models to the selected device (CPU or GPU), and prints architectural summaries using torchsummary.

In [ ]:
netG = UNetGenerator(input_channels=1, output_channels=2)
netD = PatchGANDiscriminator(input_channels=3)

netG.apply(initialize_weights)
netD.apply(initialize_weights)

netG = netG.to(device)
netD = netD.to(device)

print('GENERATOR ARCHITECTURE'.center(63))
summary(netG, (1, 256, 256))

print('DISCRIMINATOR ARCHITECTURE'.center(63))
summary(netD, [(1, 256, 256), (2, 256, 256)])

## **Optimizer and loss function setup**
Initializes hyperparameters such as learning rate, optimizer parameters (Adam), L2 regularization to prevent overfitting, and loss functions (BCEWithLogitsLoss for GAN loss and L1 for reconstruction). Also defines real and fake label smoothing values.

In [ ]:
# Learning rate
lr = 0.0002
initial_lr = lr
min_lr = 0.00008  
 
# Adam optimizer parameters
beta1 = 0.5
beta2 = 0.999

# L2 Regularization (weight decay)
weight_decay = 0.0001  #Typical values from 1e-5 to 1e-3
 
# Optimizers
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, beta2), weight_decay=weight_decay)
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, beta2), weight_decay=weight_decay)
 
# Loss weighting
LAMBDA = 100
 
# Loss functions
criterion_GAN = nn.BCEWithLogitsLoss()
criterion_L1 = nn.L1Loss()
#gen_loss = criterion_GAN + (LAMBDA * criterion_L1)

#Label smoothing
real_label = 0.9 
fake_label = 0.1

## **Directory creation for outputs**
Creates directories to store generated images, training plots, and model checkpoints. This ensures outputs are organized and saved during training.

In [ ]:
#Folder for images generated during training phase
gen_img_save_path = '/kaggle/working/output/generated_img/'
os.makedirs(gen_img_save_path, exist_ok=True)

#Folder for plots 
plot_save_training_path = '/kaggle/working/output/plot_img/'
os.makedirs(plot_save_training_path, exist_ok=True)

#Folder for checkpoints
checkpoint_dir = '/kaggle/working/checkpoints/'
os.makedirs(checkpoint_dir, exist_ok=True)

print("Folders loading completed with success.")

## **LAB to RGB conversion**
Defines the lab_tensors_to_rgb function to convert LAB image tensors (L and AB channels) into RGB format. Includes error handling for shape mismatches and invalid values like NaNs or infinities.

In [ ]:
def lab_tensors_to_rgb(l_tensor, ab_tensor):
    try:
        l = l_tensor.squeeze().cpu().numpy() * 100.0  # [0,1] -> [0,100]
        ab = ab_tensor.squeeze().detach().cpu().numpy() * 128.0  # [-1,1] -> [-128,128]

        if ab.shape[0] == 2 and ab.ndim == 3:
            ab = ab.transpose(1, 2, 0)

        if l.ndim == 3:
            l = l.squeeze()

        if l.shape != ab.shape[:2]:
            raise ValueError(f"Dimensions of L {l.shape} and AB {ab.shape} are not the same.")

        if np.isnan(l).any() or np.isinf(l).any() or np.isnan(ab).any() or np.isinf(ab).any():
            print("Invalid values in L or AB!")
            return np.zeros((l.shape[0], l.shape[1], 3), dtype=np.float32)

        l = np.expand_dims(l, axis=-1)  # [H, W, 1]
        lab = np.concatenate([l, ab], axis=2)  # [H, W, 3]

        rgb = lab2rgb(lab)
        rgb = np.clip(rgb, 0, 1)

        return rgb.astype(np.float32)

    except Exception as e:
        print(f"Error in lab_tensors_to_rgb: {e}")
        return np.zeros((256, 256, 3), dtype=np.float32)

## **Quantitative metrics**
Implements the calculate_metrics function to evaluate colorization results using metrics such as PSNR, SSIM, DeltaE and chroma accuracy. Includes error handling to skip corrupted or invalid data.

| **Metric**            | **Desirable Value**             | **Description**                                                                 |
|-----------------------|----------------------------------|---------------------------------------------------------------------------------|
| **PSNR**              | > **25 dB** (preferably >30)     | Higher values indicate lower pixel-by-pixel difference from the original image. |
| **SSIM**              | > **0.85** (preferably >0.9)     | Measures structural similarity. 1 means identical, 0 means completely different. |
| **DeltaE**            | < **5** (ideal <2)               | Measures color difference. <2 is imperceptible to the human eye, 2–5 is acceptable. |
| **Chroma Accuracy**   | Close to **1**                   | Depends on the formula. Here it measures accuracy, not error.                    |


In [ ]:
def calculate_metrics(fake_rgb, real_rgb):
    psnr_vals, ssim_vals, delta_e_vals, chroma_accs = [], [], [], []

    for fake, real in zip(fake_rgb, real_rgb):
        try:
            if np.isnan(fake).any() or np.isnan(real).any():
                print("Skip image with NaN.")
                continue

            psnr_vals.append(psnr(real, fake, data_range=1.0))
            ssim_vals.append(ssim(real, fake, channel_axis=2, data_range=1.0))

            lab_fake = rgb2lab(fake)
            lab_real = rgb2lab(real)

            delta_e_vals.append(np.mean(deltaE_cie76(lab_real, lab_fake)))

            chroma_fake = np.linalg.norm(lab_fake[..., 1:], axis=2)
            chroma_real = np.linalg.norm(lab_real[..., 1:], axis=2)
            chroma_fake = np.clip(chroma_fake, 0, chroma_real.max())  
            denom = np.maximum(chroma_real, 1e-3)  
            chroma_accs.append(np.mean(np.exp(-np.abs(chroma_fake - chroma_real) / denom)))

        except Exception as e:
            print(f"Error in calculating metrics: {e}")
            continue

    return {
        'PSNR': np.mean(psnr_vals),
        'SSIM': np.mean(ssim_vals) if ssim_vals else float('nan'),
        'DeltaE': np.mean(delta_e_vals) if delta_e_vals else float('nan'),
        'Chroma_Accuracy': np.mean(chroma_accs) if chroma_accs else float('nan')
    }

## **Loss curve plotting**
Defines a function to plot and save the generator and discriminator loss curves over training iterations. This helps visualize GAN convergence and training dynamics.

In [ ]:
#Function to plot and save loss curves of generator and discriminator
def plot_and_save_losses(G_losses, D_losses, epoch, plot_save_training_path):
    plt.figure(figsize=(10, 5))
    plt.title("Generator and discriminator losses during training")
    plt.plot(G_losses, label="Generator")
    plt.plot(D_losses, label="Discriminator")
    plt.xlabel("Iterations")
    plt.ylabel("Loss")
    plt.legend()

    loss_plot_path = os.path.join(plot_save_training_path, f'epoch_{epoch}_loss_plot.png')
    plt.savefig(loss_plot_path)
    plt.show()
    plt.close()
    print(f"Loss plot saved at {loss_plot_path}")

## **Metric plotting during training**
Implements a function to plot and save training progress of different image quality metrics (PSNR, SSIM, etc.) over epochs, providing insights into model performance over time.

In [ ]:
#Function to plot and save metrics
def plot_and_save_metrics(vec_psnr, vec_ssim, vec_delta_e, vec_chroma_acc, epoch, plot_save_training_path):

    metrics = {
        'PSNR': vec_psnr,
        'SSIM': vec_ssim,
        'DeltaE': vec_delta_e,
        'Chroma_Accuracy': vec_chroma_acc,
    }

    for metric_name, metric_values in metrics.items():
        plt.figure(figsize=(10, 5))
        plt.title(f"{metric_name} during training")
        plt.plot(metric_values, label=metric_name, color='b')
        plt.xlabel("Epochs")
        plt.ylabel(metric_name)
        plt.legend()

        metric_plot_path = os.path.join(plot_save_training_path, f'epoch_{epoch}_{metric_name.lower().replace(" ", "_")}_plot.png')
        plt.savefig(metric_plot_path)
        #plt.show()
        plt.close()  
        print(f"{metric_name} plot saved at {metric_plot_path}")

## **Checkpoint loader**
Searches for the most recent saved model checkpoint. If found, it loads the generator (netG), discriminator (netD), optimizers, training history (losses, metrics, images), and sets the training to resume from the correct epoch. If not found, training starts from scratch.

In [ ]:
# === Checkpoint loader ===
checkpoint_dataset_dir = '/kaggle/input/checkpoints-dataset'

def find_latest_checkpoint():
    checkpoints = glob.glob(os.path.join(checkpoint_dataset_dir, 'checkpoint_epoch_*.pth'))
    return max(checkpoints, key=lambda x: int(re.findall(r'\d+', x)[0])) if checkpoints else None

latest_checkpoint_path = find_latest_checkpoint()
if latest_checkpoint_path:
    try:
        checkpoint = torch.load(latest_checkpoint_path, map_location='cpu', weights_only=False)
        netG.load_state_dict(checkpoint['generator_state_dict'])
        netD.load_state_dict(checkpoint['discriminator_state_dict'])
        optimizerG.load_state_dict(checkpoint['generator_optimizer_state_dict'])
        optimizerD.load_state_dict(checkpoint['discriminator_optimizer_state_dict'])

        G_losses = checkpoint['G_losses']
        D_losses = checkpoint['D_losses']
        img_list = checkpoint['img_list']
        vec_psnr = checkpoint['vec_psnr']
        vec_ssim = checkpoint['vec_ssim']
        vec_delta_e = checkpoint['vec_delta_e']
        vec_chroma_acc = checkpoint['vec_chroma_acc']
        D_x = checkpoint['D_x']
        iters = checkpoint['iters']
        start_epoch = checkpoint['epoch'] + 1
        print(f"Checkpoint found: {latest_checkpoint_path}. Resuming from epoch {start_epoch}")
    except Exception as e:
        print("Failed to load checkpoint:", e)
        start_epoch, iters = 0, 0
        G_losses, D_losses, img_list = [], [], []
        vec_psnr, vec_ssim, vec_delta_e, vec_chroma_acc = [], [], [], []
else:
    print("No checkpoint found. Starting from scratch.")
    start_epoch, iters = 0, 0
    G_losses, D_losses, img_list = [], [], []
    vec_psnr, vec_ssim, vec_delta_e, vec_chroma_acc = [], [], [], []

## **Validation function**
Evaluates the generator model on the validation dataset. It calculates several image quality metrics (PSNR, SSIM, DeltaE, Chroma Accuracy), collects sample images for comparison (input, ground truth, generated), and optionally displays them. Results are averaged and printed per epoch.

In [ ]:
def validate(netG, val_loader, device, criterion_L1, lab_tensors_to_rgb, calculate_metrics,
             epoch, max_images_to_show=5, show_images=True):
    netG.eval()
    global count
    
    val_metrics = {'PSNR': [], 'SSIM': [], 'DeltaE': [], 'Chroma_Accuracy': []}
    fake_vs_real_imgs = []
    shown = 0

    with torch.no_grad():
        for val_l, val_ab in tqdm(val_loader, desc=f"[Validation - Epoch {epoch}]"):
            val_l, val_ab = val_l.to(device), val_ab.to(device)
            fake_ab = netG(val_l)

            real_rgb = [lab_tensors_to_rgb(l.cpu(), ab.cpu()) for l, ab in zip(val_l, val_ab)]
            fake_rgb = [lab_tensors_to_rgb(l.cpu(), ab.cpu()) for l, ab in zip(val_l, fake_ab)]

            metrics = calculate_metrics(fake_rgb, real_rgb)
            for k in val_metrics:
                val_metrics[k].append(metrics[k])

            if shown < max_images_to_show:
                for i in range(val_l.size(0)):
                    if shown >= max_images_to_show:
                        break

                    l_img = val_l[i].cpu().squeeze(0).numpy()
                    l_img_3ch = np.stack([l_img]*3, axis=0)
                    l_tensor = torch.from_numpy(l_img_3ch)

                    fake_tensor = torch.from_numpy(fake_rgb[i]).permute(2, 0, 1)
                    real_tensor = torch.from_numpy(real_rgb[i]).permute(2, 0, 1)

                    row = torch.cat([l_tensor, real_tensor, fake_tensor], dim=2)
                    fake_vs_real_imgs.append(row)
                    shown += 1

    for k in val_metrics:
        val_metrics[k] = np.nanmean(val_metrics[k])
        print(f"{k}: {val_metrics[k]:.4f}")  

    if show_images and fake_vs_real_imgs:
        grid = torch.cat(fake_vs_real_imgs, dim=1)
        plt.figure(figsize=(12, 2 * max_images_to_show))
        plt.axis("off")
        plt.title(f"Validation epoch {epoch}\nInput (B/W) | Target (Real) | Output (Generated)")
        plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
        plt.show()
        plt.close()

    return val_metrics

## **Training loop**
Training of the GAN model by running the generator and discriminator over multiple epochs. It includes techniques such as label flipping, label smoothing, and linear learning rate decay starting from epoch 60. During training, various image quality metrics (PSNR, SSIM, DeltaE, Chroma Accuracy) are collected. Every few epochs, previews of the colorized images are generated, loss and metric plots are created, validation is performed on the validation set and model checkpoints are saved. This ensures that the model's progress is trackable, visualizable, and recoverable:
- **PSNR**, **SSIM** and **Chroma Accuracy** should increase over time;
- **DeltaE** should decrease;
- Save generated images every N epochs to allow visual comparison.

In [ ]:
# === Training Loop ===
num_epochs = 51
batch_size = 16 
update_D_every = 2
flip_prob = 0.05

for epoch in range(start_epoch, num_epochs):

    # === Linear Learning Rate Decay (epochs 35-50) ===
    if epoch >= 35:
        decayed_lr = min_lr + (initial_lr - min_lr) * (1 - (epoch - 35) / (50-35))  # 15 epochs of decay
        for param_group in optimizerG.param_groups:
            param_group['lr'] = decayed_lr
        for param_group in optimizerD.param_groups:
            param_group['lr'] = decayed_lr
        print(f"[Epoch {epoch}] Learning rate decayed to {decayed_lr:.6f}")
    
    netG.train()
    netD.train()
    
    # Training loop
    for i, (real_l, real_ab) in enumerate(tqdm(train_loader, desc=f"[Training - Epoch {epoch}]")):
        real_l, real_ab = real_l.to(device), real_ab.to(device)
        b_size = real_l.size(0)
 
        # === LABEL FLIPPING (5%) + LABEL SMOOTHING ===
        if torch.rand(1).item() < flip_prob:
            real_label_val, fake_label_val = 0.1, 0.9
        else:
            real_label_val, fake_label_val = 0.9, 0.1
 
        # === Train Discriminator ===
        if i % update_D_every == 0:
            netD.zero_grad()
 
            output_real = netD(real_l, real_ab)
            label_real = torch.full_like(output_real, real_label_val, device=device)
            errD_real = criterion_GAN(output_real, label_real)
            errD_real.backward()
            D_x = output_real.mean().item()
    
            fake_ab = netG(real_l) 
            output_fake = netD(real_l, fake_ab.detach())
            label_fake = torch.full_like(output_fake, fake_label_val, device=device)
            errD_fake = criterion_GAN(output_fake, label_fake)
            errD_fake.backward()
    
            errD = errD_real + errD_fake
            optimizerD.step()
        else:
            fake_ab = netG(real_l) 
 
        # === Train Generator ===
        netG.zero_grad()
        output_fake_for_G = netD(real_l, fake_ab)
 
        #GAN loss
        label_gen = torch.full_like(output_fake_for_G, real_label, device=device) # sempre soft-real
        loss_gan = criterion_GAN(output_fake_for_G, label_gen)
        train_loss_l1 = criterion_L1(fake_ab, real_ab)

        errG = loss_gan + (LAMBDA * train_loss_l1)
        
        errG.backward()
        optimizerG.step()
 
        G_losses.append(errG.item())
        D_losses.append(errD.item() if i % update_D_every == 0 else D_losses[-1])
        
        real_rgb = [lab_tensors_to_rgb(l.cpu(), ab.cpu()) for l, ab in zip(real_l, real_ab)]
        fake_rgb = [lab_tensors_to_rgb(l.cpu(), ab.cpu()) for l, ab in zip(real_l, fake_ab)]
        
        metrics = calculate_metrics(fake_rgb, real_rgb)
        vec_psnr.append(metrics['PSNR'])
        vec_ssim.append(metrics['SSIM'])
        vec_delta_e.append(metrics['DeltaE'])
        vec_chroma_acc.append(metrics['Chroma_Accuracy'])
    
    # Create a grid of images from the last batch for visualization
    if i == len(train_loader) - 1:
        with torch.no_grad():
            fake_ab_vis = netG(real_l).detach().cpu()
        fake_rgb_batch = [lab_tensors_to_rgb(l.cpu(), ab.cpu()) for l, ab in zip(real_l, fake_ab_vis)]
        fake_rgb_tensor = torch.stack([torch.from_numpy(img).permute(2, 0, 1) for img in fake_rgb_batch])
        img_grid = vutils.make_grid(fake_rgb_tensor[:32], nrow=8, normalize=True)
        img_list.append(img_grid)
        save_image(fake_ab_vis[:32], os.path.join(plot_save_training_path, f'{epoch}-{iters}.png'), nrow=8, normalize=True)
    
    iters += 1
    
    # End of epoch visualizations
    # 1. Training metrics       
    print(f"PSNR: {np.nanmean(vec_psnr):.4f}")
    print(f"SSIM: {np.nanmean(vec_ssim):.4f}")
    print(f"DeltaE: {np.nanmean(vec_delta_e):.4f}")    
    print(f"Chroma Accuracy: {np.nanmean(vec_chroma_acc):.4f}")    
    print(f"Generator Loss: {errG.item():.4f}")
    print(f"Discriminator Loss: {errD.item():.4f}")

    # 2. Training images visualization (every 5 epochs)
    if epoch % 5 == 0 or epoch == num_epochs - 1:
        num_vis = min(5, len(real_l)) 
        rows = []
        for idx in range(num_vis):
            # --- INPUT GRAYSCALE ---
            l_img = real_l[idx].cpu().squeeze(0).numpy()  
            l_img_3ch = np.stack([l_img]*3, axis=0) 
            l_tensor = torch.from_numpy(l_img_3ch)
            
            # --- REAL COLOR ---
            real_tensor = torch.from_numpy(real_rgb[idx]).permute(2, 0, 1)
            
            # --- FAKE COLOR ---
            fake_tensor = torch.from_numpy(fake_rgb_batch[idx]).permute(2, 0, 1)        
            
            row = torch.cat([l_tensor, real_tensor, fake_tensor], dim=2) 
            rows.append(row)
        
        grid = torch.cat(rows, dim=1)  
        
        plt.figure(figsize=(12, 2 * num_vis))
        plt.axis("off")
        plt.title(f"Training epoch {epoch}\nInput (B/W)  | Target (Real) | Output (Generated)")
        plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
        plt.show()
        img_list.append(grid)
    
    # 3. Plot training loss/metrics (every 5 epochs)
    if epoch % 5 == 0 and epoch > 0:
        plot_and_save_losses(G_losses, D_losses, epoch, plot_save_training_path)
        plot_and_save_metrics(
            vec_psnr, vec_ssim, vec_delta_e,
            vec_chroma_acc, epoch, plot_save_training_path
        )
    
    # 4. Run validation and show validation metrics/images
    if epoch % 5 == 0:
        validate(netG, val_loader, device, criterion_L1, lab_tensors_to_rgb, calculate_metrics, epoch)
    
    # 5.Save model checkpoint (every 5 epochs)
    if (epoch % 5 == 0 and epoch > 0) or epoch == num_epochs - 1:  
        checkpoint_path = os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch}.pth')
        torch.save({
            'epoch': epoch,
            'generator_state_dict': netG.state_dict(),
            'discriminator_state_dict': netD.state_dict(),
            'generator_optimizer_state_dict': optimizerG.state_dict(),
            'discriminator_optimizer_state_dict': optimizerD.state_dict(),
            'G_losses': G_losses,
            'D_losses': D_losses,
            'img_list': img_list,
            'vec_psnr': vec_psnr,
            'vec_ssim': vec_ssim,
            'vec_delta_e': vec_delta_e,
            'vec_chroma_acc': vec_chroma_acc,
            'D_x': D_x,
            'iters': iters,
        }, checkpoint_path)
        print(f"Checkpoint saved: {checkpoint_path}")
    
    # Save generated images (every 25 epochs)
    if epoch % 25 == 0:
        save_epoch_path = os.path.join(gen_img_save_path, f'epoch_{epoch}')
        os.makedirs(save_epoch_path, exist_ok=True)
 
        with torch.no_grad():
            for i in range(batch_size):
                if i < real_l.size(0):
                    l_sample = real_l[i].unsqueeze(0)
                    fake_ab_sample = netG(l_sample).cpu()
                    fake_rgb = lab_tensors_to_rgb(l_sample.cpu(), fake_ab_sample)
 
                    save_image(torch.from_numpy(fake_rgb).permute(2, 0, 1),
                               os.path.join(save_epoch_path, f'image_{i}.png'),
                               normalize=True)
        
        print(f"Generated images saved for epoch {epoch} at {save_epoch_path}")

## **Testing**
Evaluation of the GAN on test images, calculating quality metrics (PSNR, SSIM, DeltaE, etc.) and saving image grids showing input, target, and generated outputs. The metrics are then saved to a CSV file and printed for quantitative and visual performance analysis.

In [ ]:
test_output_dir = '/kaggle/working/output/test_output/'
os.makedirs(test_output_dir, exist_ok=True)

netG.eval()

test_psnr, test_ssim = [], []
test_delta_e, test_chroma_acc = [], []

# === Test loop ===
with torch.no_grad():
    for batch_idx, (real_l, real_ab) in enumerate(tqdm(test_loader, desc="[Test]")):
        real_l = real_l.to(device)
        real_ab = real_ab.to(device)

        fake_ab = netG(real_l)

        real_rgb = [lab_tensors_to_rgb(l.cpu(), ab.cpu()) for l, ab in zip(real_l, real_ab)]
        fake_rgb = [lab_tensors_to_rgb(l.cpu(), ab.cpu()) for l, ab in zip(real_l, fake_ab)]

        metrics = calculate_metrics(fake_rgb, real_rgb)
        test_psnr.append(metrics['PSNR'])
        test_ssim.append(metrics['SSIM'])
        test_delta_e.append(metrics['DeltaE'])
        test_chroma_acc.append(metrics['Chroma_Accuracy'])

        rows = []
        for i in range(min(len(fake_rgb), 8)):  
            l_img = real_l[i].cpu().squeeze(0).numpy()  
            l_img_3ch = np.stack([l_img]*3, axis=0)    
            l_tensor = torch.from_numpy(l_img_3ch)

            real_tensor = torch.from_numpy(real_rgb[i]).permute(2, 0, 1)  

            fake_tensor = torch.from_numpy(fake_rgb[i]).permute(2, 0, 1)

            row = torch.cat([l_tensor, real_tensor, fake_tensor], dim=2)
            rows.append(row)

        grid = torch.cat(rows, dim=1)
        save_image(grid, os.path.join(test_output_dir, f'batch_{batch_idx}_grid.png'), normalize=True)

print("\n--- TEST FINAL RESULTS ---")
print(f"PSNR: {np.nanmean(test_psnr):.4f}")
print(f"SSIM: {np.nanmean(test_ssim):.4f}")
print(f"DeltaE: {np.nanmean(test_delta_e):.4f}")
print(f"Chroma Accuracy: {np.nanmean(test_chroma_acc):.4f}")

df_metrics = pd.DataFrame({
    'PSNR': test_psnr,
    'SSIM': test_ssim,
    'DeltaE': test_delta_e,
    'Chroma Accuracy': test_chroma_acc
})
df_metrics.to_csv(os.path.join(test_output_dir, 'test_metrics.csv'), index=False)
print("Metrics saved in: test_metrics.csv")

plt.figure(figsize=(12, 6))
plt.axis("off")
plt.title(f"Test\nInput (B/W) | Target (Real) | Output (Generated)")
plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
plt.show()
plt.close()